# 04 — Consolidação dos alvos diários para previsão

## 1. Objetivo e escopo

Preparar um dataset diário auditável para prever **máximo e P95 do dia seguinte** no IX.br São Paulo. Média e P99 são indicadores complementares. A série soma entrada e saída e está expressa em Gbps; não representa isoladamente a capacidade utilizável da infraestrutura.

O notebook consolida 2023 (desenvolvimento) e 2024 (validação), preservando todo o calendário. Não utiliza 2025 para análise ou seleção; esse ano permanece reservado à avaliação final, com a limitação já documentada de inspeção exploratória anterior. Não implementa modelos nem baselines. A análise detalhada do horário dos máximos fica para depois.

**Reaproveitamento:** o 00 fornece o CSV processado; 01/02 já descrevem níveis, evolução e picos; 03 identifica periodicidades; 03a estabelece os diagnósticos de completude e a sensibilidade dos indicadores. Reutilizam-se aqui as agregações e convenções, sem gráficos ou repetição dessas análises. A execução independe do estado de outros notebooks.

Entrada: `data/processed/serie_temporal_filtrada_gbps.csv`. O XML e a documentação são consultados somente para auditar a referência temporal. Saídas: `data/processed/alvos_diarios_2023_2024.csv` e um JSON de metadados com regras, esquema, evidência temporal e identificador SHA-256 da entrada. Antes de atualizar as saídas, as versões anteriores são preservadas em `data/backups`.

## 2. Convenções e elegibilidade

- Dias com **pelo menos 260 das 288 posições válidas** são elegíveis por cobertura (90,278%). O critério é provisório: o 03a encontrou suporte condicionado a lacunas dispersas, mas perdas relevantes em alta demanda. Cobertura não certifica o máximo nem os percentis.
- Média, máximo e quantis são calculados diretamente sobre as observações disponíveis, **sem interpolação temporal**. P95/P99 usam `np.quantile(method="linear")`.
- Negativos, não numéricos e não finitos são ausentes; zeros são preservados. Duplicatas são contadas antes da consolidação descritiva pela média, consistente com 03a. Como convenção conservadora, dias com duplicatas ou posições fora da grade ficam inelegíveis até revisão, mesmo com cobertura suficiente. Essa regra não muda dias do recorte atual, onde tais problemas não foram encontrados.
- `*_observado_gbps` preserva o indicador medido mesmo em dias inelegíveis. `alvo_*_gbps` fica ausente nesses dias. Dias vazios continuam no calendário; não são dias com tráfego zero.
- Máximos elevados não são removidos. Razões segundo máximo/máximo e mediana da vizinhança/máximo, além da proximidade das lacunas, são diagnósticos descritivos, não rótulos de erro.
- `datahora` é o início civil em `America/Sao_Paulo`, sem fuso para agrupamento; `timestamp_inicio` e `timestamp_fim` são instantes UTC explícitos. Os horários auxiliares dos máximos indicam o início civil dos intervalos, não sua disponibilidade. Ver [decisão temporal](../DECISOES_RECORTE_TEMPORAL.md#referencia-temporal-dos-intervalos-rrd).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

base_dir = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "data/processed/serie_temporal_filtrada_gbps.csv").exists()), None)
if base_dir is None:
    raise FileNotFoundError("Entrada do notebook 00 não encontrada.")
arquivo = base_dir / "data/processed/serie_temporal_filtrada_gbps.csv"
ANOS_ANALISE = (2023, 2024)
MIN_OBS_DIA = 260
OBS_ESPERADAS = 288
INDICADORES = ["media", "p95", "p99", "maximo"]
ROTULOS = {"media": "Média", "p95": "P95", "p99": "P99", "maximo": "Máximo"}
DIAS_SEMANA = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]


# Filtra o recorte antes de converter valores ou calcular estatísticas.
partes = []
for parte in pd.read_csv(arquivo, usecols=["datahora", "total_gbps", "timestamp_inicio", "timestamp_fim"],
                         dtype={"datahora": "string"}, chunksize=200_000):
    partes.append(parte.loc[parte["datahora"].str[:4].isin(["2023", "2024"])].copy())
df = pd.concat(partes, ignore_index=True)
df["datahora"] = pd.to_datetime(df["datahora"], errors="raise")
if df["datahora"].dt.tz is not None:
    raise ValueError("Fuso explícito: revisar a grade diária antes de usar 288 posições fixas.")
df["timestamp_inicio"] = pd.to_datetime(df["timestamp_inicio"], utc=True, errors="raise")
df["timestamp_fim"] = pd.to_datetime(df["timestamp_fim"], utc=True, errors="raise")
assert df["timestamp_inicio"].is_unique and df["timestamp_fim"].is_unique
assert (df["timestamp_fim"] - df["timestamp_inicio"]).eq(pd.Timedelta(minutes=5)).all()
assert df["datahora"].eq(
    df["timestamp_inicio"].dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)).all()
valores = pd.to_numeric(df["total_gbps"], errors="coerce")
df["valor_invalido"] = ~np.isfinite(valores) | valores.lt(0)
df["total_gbps"] = valores.mask(df["valor_invalido"])
df = df.sort_values("datahora")
df["data"] = df["datahora"].dt.normalize()
df["duplicata_excedente"] = df["datahora"].duplicated()
assert set(df["datahora"].dt.year.unique()) == set(ANOS_ANALISE)
deslocamentos = (df.loc[df["datahora"].dt.year.eq(2023), "datahora"]
                 - df.loc[df["datahora"].dt.year.eq(2023), "datahora"].dt.normalize()) % pd.Timedelta(minutes=5)
if deslocamentos.nunique() != 1:
    raise ValueError("Não há deslocamento único na grade de 2023; revisar timestamps.")
offset_grade = deslocamentos.iloc[0]
assert offset_grade == pd.Timedelta(0), "Esperado início civil na grade exata de cinco minutos."
df["fora_grade"] = ((df["datahora"] - df["data"]) % pd.Timedelta(minutes=5)).ne(offset_grade)
serie = df.loc[~df["fora_grade"]].groupby("datahora")["total_gbps"].mean().sort_index()
calendario = pd.date_range("2023-01-01", "2024-12-31", freq="D", name="data")
print(f"Recorte: {df['datahora'].min()} a {df['datahora'].max()}")
print(f"Fuso: {df['datahora'].dt.tz}; deslocamento da grade: {offset_grade}.")
display(df.groupby(df["datahora"].dt.year).agg(
    registros=("datahora", "size"), invalidos=("valor_invalido", "sum"),
    duplicatas_excedentes=("duplicata_excedente", "sum"), fora_grade=("fora_grade", "sum")))

Recorte: 2023-01-01 00:00:00 a 2024-12-31 23:55:00
Fuso: None; deslocamento da grade: 0 days 00:00:00.


,registros,invalidos,duplicatas_excedentes,fora_grade
datahora,,,,
2023,105120,452,0,0
2024,105408,4217,0,0


## 3. Auditoria da referência temporal integrada

O notebook 00 confere todos os epochs de fim contra o XML. Aqui verificamos também a sequência integral dos epochs da primeira RRA e o alinhamento de suas bordas ao `lastupdate`. Em seis posições escolhidas previamente de 2023–2024, conferimos o fim UTC, o início civil e a correspondência dos valores com o CSV. Os comentários civis não são usados como referência primária; suas divergências históricas estão registradas na [decisão temporal](../DECISOES_RECORTE_TEMPORAL.md#referencia-temporal-dos-intervalos-rrd).

A primeira RRA é `AVERAGE`, `step=300`, `pdp_per_row=1`. Seus timestamps indicam o fim exclusivo de intervalos de cinco minutos. O último intervalo do dia tem fim na meia-noite seguinte e é atribuído ao dia de seu início. A auditoria não usa valores de tráfego de 2025; `lastupdate` é metadado estrutural.

In [2]:
import re

xml_path = base_dir / "data/raw/ptt.xml"
amostras_xml = []
n_linhas_xml = 0
cabecalho = []
primeira_database = False
primeiro_epoch_xml = ultimo_epoch_xml = None
with xml_path.open() as f:
    for linha in f:
        if not primeira_database:
            cabecalho.append(linha)
            if "<database>" in linha:
                primeira_database = True
            continue
        if "</database>" in linha:
            break
        if "<row>" not in linha:
            continue
        epoch = int(re.search(r"/ (\d+) -->", linha)[1])
        if primeiro_epoch_xml is None:
            primeiro_epoch_xml = epoch
        else:
            assert epoch - ultimo_epoch_xml == 300
        ultimo_epoch_xml = epoch
        m = re.search(r"<!-- (202[34]-(?:01|06|12)-01 12:00:00) (\S+) / (\d+) -->", linha)
        if m:
            valores_xml = [float(v) for v in re.findall(r"<v>(.*?)</v>", linha)]
            amostras_xml.append((n_linhas_xml, m[1], m[2], epoch, sum(valores_xml)/1e9))
        n_linhas_xml += 1
cabecalho = "".join(cabecalho)
step_xml = int(re.search(r"<step>(\d+)</step>", cabecalho)[1])
lastupdate_xml = int(re.search(r"<lastupdate>(\d+)</lastupdate>", cabecalho)[1])
pdp_por_linha = int(re.search(r"<pdp_per_row>(\d+)</pdp_per_row>", cabecalho)[1])
cf_xml = re.search(r"<cf>(.*?)</cf>", cabecalho)[1]
assert step_xml == 300 and pdp_por_linha == 1 and cf_xml == "AVERAGE"
assert len(amostras_xml) == 6
intervalo_xml = step_xml * pdp_por_linha
ultimo_fim_xml = lastupdate_xml - lastupdate_xml % intervalo_xml
primeiro_fim_xml = ultimo_fim_xml - (n_linhas_xml - 1) * intervalo_xml
assert primeiro_fim_xml == primeiro_epoch_xml and ultimo_fim_xml == ultimo_epoch_xml
csv_por_fim = df.set_index("timestamp_fim")
linhas_auditoria = []
for posicao, civil, rotulo, epoch, total_xml in amostras_xml:
    fim_utc = pd.Timestamp(epoch, unit="s", tz="UTC")
    inicio_utc = fim_utc - pd.Timedelta(seconds=intervalo_xml)
    inicio_civil = inicio_utc.tz_convert("America/Sao_Paulo").tz_localize(None)
    epoch_formula = primeiro_fim_xml + posicao * intervalo_xml
    registro_csv = csv_por_fim.loc[fim_utc]
    assert registro_csv["timestamp_inicio"] == inicio_utc
    assert registro_csv["datahora"] == inicio_civil
    civil = pd.Timestamp(civil)
    linhas_auditoria.append({"horario_xml": civil, "rotulo_xml": rotulo,
        "offset_civil_utc_h": (civil - fim_utc.tz_localize(None)).total_seconds()/3600,
        "fim_utc": fim_utc, "inicio_civil_csv": inicio_civil,
        "deslocamento_fim_s": epoch_formula - epoch,
        "valores_correspondem": bool(np.isclose(registro_csv["total_gbps"], total_xml, rtol=1e-10))})
auditoria_temporal = pd.DataFrame(linhas_auditoria)
display(auditoria_temporal)
assert auditoria_temporal["valores_correspondem"].all()
assert auditoria_temporal["deslocamento_fim_s"].eq(0).all()
assert auditoria_temporal["rotulo_xml"].eq("BRT").all()
assert auditoria_temporal["offset_civil_utc_h"].eq(-3).all()
print(f"RRA: {cf_xml}; {intervalo_xml} segundos/linha; {n_linhas_xml:,} epochs regulares.")
print("Reconstrução alinhada: fim UTC coincide com XML; dia civil atribuído pelo início.")

,horario_xml,rotulo_xml,offset_civil_utc_h,fim_utc,inicio_civil_csv,deslocamento_fim_s,valores_correspondem
0,2023-01-01 12:00:00,BRT,-3.0,2023-01-01 15:00:00+00:00,2023-01-01 11:55:00,0,True
1,2023-06-01 12:00:00,BRT,-3.0,2023-06-01 15:00:00+00:00,2023-06-01 11:55:00,0,True
2,2023-12-01 12:00:00,BRT,-3.0,2023-12-01 15:00:00+00:00,2023-12-01 11:55:00,0,True
3,2024-01-01 12:00:00,BRT,-3.0,2024-01-01 15:00:00+00:00,2024-01-01 11:55:00,0,True
4,2024-06-01 12:00:00,BRT,-3.0,2024-06-01 15:00:00+00:00,2024-06-01 11:55:00,0,True
5,2024-12-01 12:00:00,BRT,-3.0,2024-12-01 15:00:00+00:00,2024-12-01 11:55:00,0,True


RRA: AVERAGE; 300 segundos/linha; 2,100,154 epochs regulares.
Reconstrução alinhada: fim UTC coincide com XML; dia civil atribuído pelo início.


**Conclusão temporal.** Os fins estão alinhados aos epochs do XML; `datahora` representa o início civil, 300 segundos antes do fim neste recorte UTC−03:00. O erro anterior de −151 segundos foi corrigido no 00. A integração preservou integralmente contagens, elegibilidade, máximo, P95, P99 e média diária de 2023–2025, além das agregações horárias de 2023–2024. O uso de 2025 foi somente uma comparação de integridade, sem seleção metodológica.

A zona civil adotada é `America/Sao_Paulo`, com instantes preservados em UTC na entrada. Na tabela diária, os campos de horário dos máximos continuam referências civis ao início dos intervalos; a medição só pode estar disponível depois do fim e da ingestão.

## 4. Tabela diária e qualidade

`diario` tem uma linha para cada dia de 2023–2024. São preservadas contagens de posições válidas/ausentes, registros inválidos, duplicatas, zeros e posições fora da grade. As lacunas incluem as bordas do dia e são contadas em posições de cinco minutos; blocos que cruzam a meia-noite são descritos separadamente em cada dia.

As razões de robustez e a distância à ausência são calculadas em torno da **primeira ocorrência** do máximo. Também se registram última ocorrência e número de empates para rastreabilidade. Esses horários são auxiliares de auditoria; não constituem um alvo de previsão de horário nesta entrega.

`maximo_destacado` indica segundo maior valor inferior a 90% do máximo, conforme o diagnóstico do 03a, sem excluir o dia. `ausencia_ate_15min_max` indica lacuna próxima ao máximo observado. Valores ausentes nas razões permanecem ausentes: os sinalizadores isolados não certificam qualidade. A vizinhança de ±30/60 minutos é truncada no limite do dia e acompanha contagens observadas e esperadas.

In [3]:
def resumo_lacunas(ausente):
    """Blocos na grade do dia, inclusive nas bordas; cada posição representa 5 min."""
    a = np.asarray(ausente, dtype=bool)
    bordas = np.diff(np.r_[False, a, False].astype(int))
    inicios = np.flatnonzero(bordas == 1)
    fins = np.flatnonzero(bordas == -1)
    tamanhos = fins - inicios
    return {"n_blocos_ausentes": len(tamanhos),
            "maior_lacuna_min": int(tamanhos.max() * 5) if len(tamanhos) else 0,
            "lacuna_inicio_min": int(tamanhos[0] * 5) if len(tamanhos) and inicios[0] == 0 else 0,
            "lacuna_fim_min": int(tamanhos[-1] * 5) if len(tamanhos) and fins[-1] == len(a) else 0}

def indicadores_dia(s):
    """Estatísticas observadas e diagnósticos; nenhuma interpolação ou exclusão de extremos."""
    v = s.dropna()
    r = {f"{k}_observado_gbps": np.nan for k in INDICADORES}
    r.update({"instante_maximo": pd.NaT, "ultimo_instante_maximo": pd.NaT,
              "hora_maximo": np.nan, "n_empates_maximo": 0, "n_horas_maximo": 0,
              "horas_maximo": "", "segundo_max_pct_max": np.nan,
              "distancia_ausencia_max_min": np.nan})
    for minutos in [30, 60]:
        r.update({f"mediana_{minutos}min_pct_max": np.nan,
                  f"n_obs_{minutos}min": 0, f"n_esperadas_{minutos}min": 0})
    if v.empty:
        return r
    p95, p99 = np.quantile(v.to_numpy(), [.95, .99], method="linear")
    maximo = v.max()
    instantes = v.index[v.eq(maximo)]
    instante = instantes[0]
    horas = np.unique(instantes.hour)
    r.update({"media_observado_gbps": v.mean(), "p95_observado_gbps": p95,
              "p99_observado_gbps": p99, "maximo_observado_gbps": maximo,
              "instante_maximo": instante, "ultimo_instante_maximo": instantes[-1],
              "hora_maximo": instante.hour, "n_empates_maximo": len(instantes),
              "n_horas_maximo": len(horas), "horas_maximo": ",".join(map(str, horas))})
    if len(v) >= 2 and maximo > 0:
        r["segundo_max_pct_max"] = 100 * v.nlargest(2).iloc[1] / maximo
    for minutos in [30, 60]:
        janela = s.loc[instante-pd.Timedelta(minutes=minutos):instante+pd.Timedelta(minutes=minutos)]
        r[f"n_obs_{minutos}min"] = janela.count()
        r[f"n_esperadas_{minutos}min"] = len(janela)
        if maximo > 0:
            r[f"mediana_{minutos}min_pct_max"] = 100 * janela.median() / maximo
    if s.isna().any():
        r["distancia_ausencia_max_min"] = np.abs((s.index[s.isna()]-instante).total_seconds()/60).min()
    return r

registros_dia = {d: g for d, g in df.groupby("data")}
linhas = []
for data in calendario:
    g = registros_dia.get(data, df.iloc[:0])
    grade = pd.date_range(data + offset_grade, periods=OBS_ESPERADAS, freq="5min")
    s = serie.reindex(grade)
    n_validas = int(s.count())
    linha = {"data": data, "ano": data.year, "mes": data.month, "dia_semana": data.dayofweek,
             "n_registros": len(g), "n_esperadas": OBS_ESPERADAS,
             "n_validas": n_validas, "n_ausentes": OBS_ESPERADAS-n_validas,
             "n_posicoes_sem_registro": int((~grade.isin(g["datahora"])).sum()),
             "n_valores_invalidos": int(g["valor_invalido"].sum()),
             "n_duplicatas": int(g["duplicata_excedente"].sum()),
             "n_fora_grade": int(g["fora_grade"].sum()), "n_zeros": int(s.eq(0).sum()),
             "completude_pct": 100*n_validas/OBS_ESPERADAS}
    linha.update(resumo_lacunas(s.isna()))
    linha.update(indicadores_dia(s))
    linhas.append(linha)
diario = pd.DataFrame(linhas).set_index("data")
diario["estrutura_valida"] = diario["n_duplicatas"].eq(0) & diario["n_fora_grade"].eq(0)
diario["cobertura_90"] = diario["n_validas"].ge(MIN_OBS_DIA)
diario["elegivel_magnitude"] = diario["estrutura_valida"] & diario["cobertura_90"]
diario["grupo_completude"] = np.select(
    [diario["n_validas"].ge(274), diario["cobertura_90"]], ["95% ou mais", "90% a <95%"], default="Abaixo de 90%")
diario["maximo_destacado"] = diario["segundo_max_pct_max"].lt(90)
diario["ausencia_ate_15min_max"] = diario["distancia_ausencia_max_min"].le(15)
diario["horario_ambiguo"] = diario["n_horas_maximo"].gt(1)
for indicador in INDICADORES:
    diario[f"alvo_{indicador}_gbps"] = diario[f"{indicador}_observado_gbps"].where(diario["elegivel_magnitude"])
colunas_alvos = [f"alvo_{k}_gbps" for k in INDICADORES]
validos = diario.loc[diario["elegivel_magnitude"]].copy()
display(diario[colunas_alvos + ["n_validas", "completude_pct", "elegivel_magnitude"]].head().round(2))
print(f"Dataset: {len(diario)} dias e {len(diario.columns)} colunas.")

,alvo_media_gbps,alvo_p95_gbps,alvo_p99_gbps,alvo_maximo_gbps,n_validas,completude_pct,elegivel_magnitude
data,,,,,,,
2023-01-01,2833.02,4519.48,4594.80,7483.38,288,100.00,True
2023-01-02,2766.03,4350.23,4455.80,4478.52,288,100.00,True
2023-01-03,2724.35,4273.37,4354.20,4379.92,288,100.00,True
2023-01-04,2658.34,4137.23,4219.18,6103.66,287,99.65,True
2023-01-05,2549.65,3832.39,3888.57,3912.14,288,100.00,True


Dataset: 731 dias e 46 colunas.


## 5. Conferências compactas

As tabelas conferem disponibilidade e ordem de grandeza, sem refazer a exploração anterior. As estatísticas de magnitude usam somente os dias elegíveis, os mesmos para os quatro indicadores. A tabela mensal mantém a visibilidade dos períodos com perda de alvos. Os 362 dias de 2023 e 347 de 2024 devem coincidir com o 03a para esta versão da entrada.

In [4]:
disponibilidade = diario.groupby("ano").agg(
    dias_calendario=("n_validas", "size"), dias_completos=("n_validas", lambda x: x.eq(288).sum()),
    dias_elegiveis=("elegivel_magnitude", "sum"),
    dias_duplicados=("n_duplicatas", lambda x: x.gt(0).sum()),
    dias_fora_grade=("n_fora_grade", lambda x: x.gt(0).sum()))
disponibilidade["dias_sem_alvo"] = disponibilidade["dias_calendario"]-disponibilidade["dias_elegiveis"]
display(disponibilidade)
disponibilidade_mensal = diario.groupby(["mes", "ano"])["elegivel_magnitude"].agg(["sum", "size"])
display(disponibilidade_mensal.rename(columns={"sum": "elegiveis", "size": "dias"}).unstack("ano"))
display(validos.groupby("ano")[colunas_alvos].agg(["count", "median", "min", "max"]).T.round(2))

,dias_calendario,dias_completos,dias_elegiveis,dias_duplicados,dias_fora_grade,dias_sem_alvo
ano,,,,,,
2023,365,252,362,0,0,3
2024,366,20,347,0,0,19


elegiveis      dias     
ano      2023 2024 2023 2024
mes                         
1          31   31   31   31
2          28   20   28   29
3          31   23   31   31
4          29   29   30   30
5          31   31   31   31
6          30   30   30   30
7          31   30   31   31
8          31   31   31   31
9          30   30   30   30
10         31   31   31   31
11         29   30   30   30
12         30   31   31   31

ano                          2023      2024
alvo_media_gbps  count     362.00    347.00
                 median   2893.42   3248.52
                 min      2238.42   2845.47
                 max      3930.80   4186.26
alvo_p95_gbps    count     362.00    347.00
                 median   4698.47   5358.01
                 min      3156.88   3966.50
                 max      7589.62   8167.37
alvo_p99_gbps    count     362.00    347.00
                 median   4795.54   5481.68
                 min      3168.51   4006.22
                 max     11222.50   9156.62
alvo_maximo_gbps count     362.00    347.00
                 median   4877.87   5643.54
                 min      3176.18   4045.37
                 max     11919.24  16434.22

## 6. Validação da construção

Verificam-se calendário, quantis, elegibilidade e reconciliação com a cobertura anterior. Casos pequenos verificam lacunas nas bordas, dias vazios e empates. Nenhum teste de previsão é realizado. As contagens de referência são explícitas para detectar mudanças na entrada, não regras para forçar um resultado.

In [5]:
assert diario.index.equals(calendario)
assert diario.index.is_unique and diario.index.is_monotonic_increasing
assert set(diario["ano"]) == {2023, 2024} and len(diario) == 731
assert diario["n_validas"].between(0, 288).all()
assert (diario["n_validas"] + diario["n_ausentes"]).eq(288).all()
assert diario.loc[~diario["elegivel_magnitude"], colunas_alvos].isna().all().all()
assert diario.loc[diario["elegivel_magnitude"], colunas_alvos].notna().all().all()
assert (validos["alvo_p95_gbps"] <= validos["alvo_p99_gbps"]).all()
assert (validos["alvo_p99_gbps"] <= validos["alvo_maximo_gbps"]).all()
# Referências explícitas para esta versão da entrada; mudanças exigem revisar a reconciliação.
assert diario.groupby("ano")["cobertura_90"].sum().to_dict() == {2023: 362, 2024: 347}
assert resumo_lacunas([True, True, False, True])["maior_lacuna_min"] == 10
assert resumo_lacunas([True, True, False, True])["lacuna_fim_min"] == 5
assert resumo_lacunas(np.ones(288, dtype=bool))["maior_lacuna_min"] == 1440
assert resumo_lacunas(np.zeros(288, dtype=bool))["n_blocos_ausentes"] == 0
s_teste = pd.Series(np.arange(1., 289.), index=pd.date_range("2023-01-01 00:00:00", periods=288, freq="5min"))
r_teste = indicadores_dia(s_teste)
np.testing.assert_allclose([r_teste["p95_observado_gbps"], r_teste["p99_observado_gbps"]], [273.65, 285.13])
s_teste.iloc[0] = 288
assert indicadores_dia(s_teste)["n_horas_maximo"] == 2
assert indicadores_dia(s_teste)["instante_maximo"] == s_teste.index[0]
assert indicadores_dia(s_teste*0)["n_empates_maximo"] == 288
assert pd.isna(indicadores_dia(s_teste*np.nan)["maximo_observado_gbps"])
print("Validação concluída: calendário, cobertura, quantis, elegibilidade, lacunas e empates consistentes.")

Validação concluída: calendário, cobertura, quantis, elegibilidade, lacunas e empates consistentes.


## 7. Exportação e uso independente no notebook 05

O CSV contém **todos os dias**, com os indicadores observados, os alvos mascarados e a qualidade. Usa UTF-8, ponto decimal, vírgula separadora e campo vazio para ausentes, conforme os CSVs processados do projeto. `data` é a chave diária, não o instante da previsão. Datas são gravadas em ISO; não há deslocamento de fuso anexado.

O JSON registra tipos, convenções, identificação da entrada, disponibilidade e a convenção temporal integrada. No 05, carregar com `pd.read_csv(..., parse_dates=["data", "instante_maximo", "ultimo_instante_maximo"])`, ordenar e usar `set_index("data").asfreq("D")`. Os alvos principais são `alvo_maximo_gbps` e `alvo_p95_gbps`; `alvo_p99_gbps` e `alvo_media_gbps` são complementares. Não eliminar dias antes de construir defasagens. As medidas e diagnósticos de D só ficam disponíveis após o encerramento de D e a ingestão: não podem ser atributos da previsão de D realizada em D−1.

O notebook 05 pode carregar o artefato de forma independente. A referência diária foi integrada e validada; permanecem por definir o horário operacional da origem de previsão e a latência de ingestão. Nenhuma regra selecionada com 2025 é introduzida.

In [6]:
import hashlib
import json
import shutil
import tempfile

saida_csv = base_dir / "data/processed/alvos_diarios_2023_2024.csv"
saida_json = saida_csv.with_suffix(".metadata.json")
backups_raiz = base_dir / "data/backups"
backups_raiz.mkdir(parents=True, exist_ok=True)
backup_dir = Path(tempfile.mkdtemp(prefix="rrd_execucao04_", dir=backups_raiz))
manifesto_backup = {}
for anterior in [saida_csv, saida_json]:
    if anterior.exists():
        digest = hashlib.sha256(anterior.read_bytes()).hexdigest()
        copia = backup_dir / anterior.name
        shutil.copy2(anterior, copia)
        assert hashlib.sha256(copia.read_bytes()).hexdigest() == digest
        manifesto_backup[anterior.name] = {"sha256": digest, "bytes": anterior.stat().st_size}
(backup_dir / "manifesto.json").write_text(json.dumps(manifesto_backup, indent=2)+"\n")
diario.to_csv(saida_csv, index=True, index_label="data", encoding="utf-8", na_rep="", date_format="%Y-%m-%d %H:%M:%S")
# Valida o artefato que o 05 efetivamente lerá, incluindo máscaras e precisão numérica.
releitura = pd.read_csv(saida_csv, parse_dates=["data", "instante_maximo", "ultimo_instante_maximo"]).set_index("data")
assert releitura.index.equals(diario.index)
assert releitura.columns.equals(diario.columns)
for coluna in diario.select_dtypes(include=["number", "bool"]).columns:
    np.testing.assert_allclose(releitura[coluna].to_numpy(dtype=float), diario[coluna].to_numpy(dtype=float),
                               rtol=1e-12, atol=1e-10, equal_nan=True)
assert releitura[colunas_alvos].isna().equals(diario[colunas_alvos].isna())
for coluna in ["instante_maximo", "ultimo_instante_maximo"]:
    assert releitura[coluna].equals(diario[coluna])
metadados = {
    "versao_esquema": 2, "notebook": "04_target_definition.ipynb",
    "entrada": str(arquivo.relative_to(base_dir)),
    "entrada_sha256": hashlib.sha256(arquivo.read_bytes()).hexdigest(),
    "dataset": str(saida_csv.relative_to(base_dir)),
    "dataset_sha256": hashlib.sha256(saida_csv.read_bytes()).hexdigest(),
    "periodo": ["2023-01-01", "2024-12-31"], "linhas": len(diario),
    "unidade": "Gbps; entrada + saída", "chave": "data",
    "alvos_principais": ["alvo_maximo_gbps", "alvo_p95_gbps"],
    "indicadores_complementares": ["alvo_p99_gbps", "alvo_media_gbps"],
    "quantil": "numpy.quantile(method='linear')", "interpolacao_temporal": False,
    "elegibilidade": "n_validas >= 260; n_duplicatas == 0; n_fora_grade == 0",
    "observacoes_esperadas_dia": 288, "min_observacoes": 260,
    "dias_preservados": "Calendário completo; alvos inelegíveis ausentes; observados preservados",
    "duplicatas": "Contadas antes da média por timestamp; dia inelegível até revisão",
    "extremos": "Sinalizadores descritivos; nenhuma remoção por magnitude",
    "empates": "Primeiro e último máximo registrados; horário não é alvo nesta versão",
    "temporal": {
        "status": "integrado_validado",
        "fuso_atribuido_csv": None,
        "referencia_datahora": "Início civil do intervalo; não é instante de disponibilidade",
        "fuso_civil_adotado": "America/Sao_Paulo", "fuso_instantes_entrada": "UTC",
        "evidencia": "Sequência integral dos epochs da primeira RRA; seis amostras de 2023–2024 conferidas com os instantes UTC e valores do CSV",
        "deslocamento_fim_vs_epoch_xml_segundos": 0,
        "deslocamento_csv_vs_linha_xml_segundos": -300,
        "erro_anterior_fim_segundos": -151,
        "atribuicao_dia": "Dia civil do início do intervalo [início, fim)",
        "grade_csv_offset": str(offset_grade), "rra_cf": cf_xml,
        "rra_step_segundos": step_xml, "pdp_per_row": pdp_por_linha,
        "limite": "Horários dos máximos são referências civis de início; disponibilidade exige fim e latência de ingestão"},
    "disponibilidade": disponibilidade.reset_index().to_dict(orient="records"),
    "colunas_tipos_pandas": {c: str(t) for c, t in diario.dtypes.items()},
    "regras_colunas": {
        "*_observado_gbps": "Estatística das medições disponíveis, inclusive dias inelegíveis",
        "alvo_*_gbps": "Mesmo valor observado somente em dias elegíveis",
        "completude_pct": "100 * n_validas / 288",
        "maior_lacuna_min": "Maior sequência ausente dentro do dia, incluindo bordas, vezes 5",
        "segundo_max_pct_max": "100 * segundo maior valor / máximo; ausente se indefinido",
        "mediana_30min_pct_max / mediana_60min_pct_max": "Razão da mediana na vizinhança do primeiro máximo, truncada no dia",
        "maximo_destacado": "segundo_max_pct_max < 90; não é classificação de erro",
        "ausencia_ate_15min_max": "Ausência a até 15 min do primeiro máximo observado",
        "horario_ambiguo": "Máximo em mais de uma faixa de hora; somente diagnóstico",
        "instante_maximo / ultimo_instante_maximo": "Referências civis ao início dos intervalos dos máximos; não indicam disponibilidade"},
    "origem_previsao": "D−1 para D; confirmar horário operacional e latência. Diagnósticos de D não disponíveis na véspera"
}
saida_json.write_text(json.dumps(metadados, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
assert json.loads(saida_json.read_text(encoding="utf-8"))["linhas"] == len(releitura)
print(f"Exportado e relido: {saida_csv.relative_to(base_dir)} ({len(releitura)} dias)")
print(f"Metadados: {saida_json.relative_to(base_dir)}")

Exportado e relido: data/processed/alvos_diarios_2023_2024.csv (731 dias)
Metadados: data/processed/alvos_diarios_2023_2024.metadata.json


## 8. Resultado e decisões pendentes

- Dataset com **731 dias**: **362 elegíveis em 2023** e **347 em 2024**. Permanecem no calendário os 3 e 19 dias sem alvo, respectivamente. Há 252 e 20 dias completos; não foram encontradas duplicatas ou posições fora da grade no recorte.
- As quatro estatísticas compartilham a mesma máscara de elegibilidade. P95 ≤ P99 ≤ máximo é verificado; média é complementar e não substitui demanda elevada. A releitura do CSV verifica datas, valores numéricos e máscaras.
- O limiar de 90% permanece **provisório**, sem garantia de captura do verdadeiro extremo. A tolerância operacional a erros de medição dos alvos ainda não foi definida; não se criam limiares diferentes por indicador nesta etapa.
- **Referência temporal integrada:** início civil em `America/Sao_Paulo`, instantes UTC explícitos na entrada e fins alinhados aos epochs RRD. Os alvos e a elegibilidade foram preservados; os horários auxiliares passaram para a grade exata de cinco minutos.
- O 05 recebe uma tabela independente e auditável, com horizonte de um dia. Confirmar horário da origem e latência de disponibilidade. Nenhum baseline, previsão ou análise detalhada do horário dos máximos foi implementado aqui.